In [ ]:
# loading libraries
import pyfesom2 as pf
import copernicusmarine
print(copernicusmarine.__version__)
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import xarray as xr
import time
from tqdm.auto import tqdm
import os
import pandas as pd

In [ ]:
#loading Copernicus data for 2014

data_request = {
    "dataset_id": "OSISAF-GLO-SEAICE_CONC_TIMESERIES-NH-LA-OBS",

    # whole Arctic domain
    "longitude": [-180, 180],
    "latitude": [60, 90],

    "time": ["2014-01-01", "2015-01-01"]
}

ds = copernicusmarine.open_dataset(
    dataset_id=data_request["dataset_id"],
    minimum_longitude=data_request["longitude"][0],
    maximum_longitude=data_request["longitude"][1],
    minimum_latitude=data_request["latitude"][0],
    maximum_latitude=data_request["latitude"][1],
    start_datetime=data_request["time"][0],
    end_datetime=data_request["time"][1],
)

print(ds)

In [ ]:
print(list(ds.data_vars))

In [ ]:
# defining ice edgeimport time
start = time.time()

ice_edge_list = []

# Calculate ice edge for each day
for i in tqdm(
    range(ds.sizes["time"]),
    desc="Calculating ice edge",
    unit="day"
):
    ds_day = ds.isel(time=slice(i, i + 1))

    edge_day = ice_edge_latitude_by_lon(
        ds_day,
        level=15
    )

    ice_edge_list.append(np.asarray(edge_day))

# Combine results along the time dimension
ice_edge = np.concatenate(ice_edge_list, axis=0)

print("Ice edge shape:", ice_edge.shape)

end = time.time()
print(f"Execution time: {end - start:.3f} seconds")

def ice_edge_latitude_by_lon(ds, sic_name="ice_conc", level=15, lon_grid=None):
    if lon_grid is None:
        lon_grid = np.arange(-180, 181, 1)

    edge_all_times = []

    for t in range(ds.sizes["time"]):
        sic = ds[sic_name].isel(time=t)

        fig, ax = plt.subplots()
        cs = ax.contour(
            ds["longitude"].values,
            ds["latitude"].values,
            sic.values,
            levels=[level]
        )
        plt.close(fig)

        lons_all = []
        lats_all = []

        for seg in cs.allsegs[0]:
            lon_seg = seg[:, 0]
            lat_seg = seg[:, 1]

            lon_seg = ((lon_seg + 180) % 360) - 180

            order = np.argsort(lon_seg)
            lon_seg = lon_seg[order]
            lat_seg = lat_seg[order]

            good = np.isfinite(lon_seg) & np.isfinite(lat_seg)
            lon_seg = lon_seg[good]
            lat_seg = lat_seg[good]

            if len(lon_seg) > 2:
                lons_all.append(lon_seg)
                lats_all.append(lat_seg)

        edge_lat = np.full(len(lon_grid), np.nan)

        for i, lon0 in enumerate(lon_grid):
            candidates = []

            for lon_seg, lat_seg in zip(lons_all, lats_all):
                if lon0 >= np.nanmin(lon_seg) and lon0 <= np.nanmax(lon_seg):
                    lat0 = np.interp(lon0, lon_seg, lat_seg)
                    candidates.append(lat0)

            if len(candidates) > 0:
                # Arctic: usually take the southernmost 15% crossing = ice edge
                edge_lat[i] = np.nanmin(candidates)

        edge_all_times.append(edge_lat)

    edge_da = xr.DataArray(
        np.array(edge_all_times),
        dims=("time", "longitude"),
        coords={
            "time": ds["time"],
            "longitude": lon_grid
        },
        name="ice_edge_latitude",
        attrs={
            "description": "Latitude where sea ice concentration crosses 15%",
            "units": "degrees_north",
            "threshold": level
        }
    )

    return edge_da

In [ ]:
start = time.time()

ice_edge_list = []

# Calculate ice edge for each day
for i in tqdm(
    range(ds.sizes["time"]),
    desc="Calculating ice edge",
    unit="day"
):
    ds_day = ds.isel(time=slice(i, i + 1))

    edge_day = ice_edge_latitude_by_lon(
        ds_day,
        level=15
    )

    ice_edge_list.append(np.asarray(edge_day))

# Combine results along the time dimension
ice_edge = np.concatenate(ice_edge_list, axis=0)

print("Ice edge shape:", ice_edge.shape)

end = time.time()
print(f"Execution time: {end - start:.3f} seconds")

In [ ]:
#save copernicus data

SAVE_DIR = "./saved_data"
os.makedirs(SAVE_DIR, exist_ok=True)

SAVE_FILE = os.path.join(SAVE_DIR, "ice_edge.npy")

np.save(SAVE_FILE, ice_edge)

print(f"Saved to: {os.path.abspath(SAVE_FILE)}")
print(f"Shape: {ice_edge.shape}")

In [ ]:
#load copernicus data back
ice_edge = np.load("./saved_data/ice_edge.npy")

print("Loaded ice edge:", ice_edge.shape)

In [ ]:
# continue with FESOM data
#loading the mesh
meshpath = '/albedo/work/projects/p_trr181/ebagaeva/work/ebagaeva/3IGW/Meshes/farc/'
alpha, beta, gamma=[0, 0, 0]
mesh=pf.load_mesh(meshpath, abg=[alpha, beta, gamma], usepickle = False, usejoblib=True)

In [ ]:
#loading the data
# load correct hourly data output

data_path= '/albedo/work/projects/p_trr181/ebagaeva/work/ebagaeva/3IGW/JRA55_2014/Tides_2014/'

a_ice = pf.get_data(data_path,
                   "a_ice", 
                   2014,
                   mesh,
                   how='ori',
                   compute=False, 
                  chunks={'nod2':10000, 'nz1':5})

print (a_ice.shape)


In [ ]:
# ============================================================
# 1. FUNCTION: FESOM SEA-ICE EDGE LATITUDE
# ============================================================

def ice_edge_latitude_by_lon_fesom(
    a_ice,
    mesh,
    level=0.15,
    min_lat=60,
    lon_grid=None,
    triangles=None,
):
    """
    Extract latitude of the FESOM sea-ice edge as a function
    of longitude.

    Parameters
    ----------
    a_ice : xarray.DataArray
        Sea ice concentration with dimensions (time, nod2).
        Usually in fraction, so 15% = 0.15.

    mesh : pyfesom2 mesh
        Must contain mesh.x2, mesh.y2, and mesh.elem.

    level : float
        Sea ice concentration threshold. Default is 0.15.

    min_lat : float
        Only use nodes north of this latitude.

    lon_grid : array-like
        Longitudes where ice-edge latitude will be interpolated.

    triangles : array-like
        Mesh triangles. If None, uses mesh.elem.

    Returns
    -------
    xarray.DataArray
        Ice-edge latitude with dimensions (time, longitude).
    """

    # ========================================================
    # LONGITUDE GRID
    # ========================================================

    if lon_grid is None:
        lon_grid = np.arange(-180, 181, 1)

    lon_grid = np.asarray(lon_grid)

    # ========================================================
    # MESH COORDINATES
    # ========================================================

    lon_nodes = np.asarray(mesh.x2)
    lat_nodes = np.asarray(mesh.y2)

    # Convert longitudes to [-180, 180)
    lon_nodes = ((lon_nodes + 180) % 360) - 180

    if triangles is None:
        triangles = np.asarray(mesh.elem)
    else:
        triangles = np.asarray(triangles)

    # Keep only Arctic nodes
    north_mask = lat_nodes >= min_lat

    # ========================================================
    # TRIANGULATION
    # ========================================================

    tri = mtri.Triangulation(
        lon_nodes,
        lat_nodes,
        triangles=triangles
    )

    edge_all_times = []

    # ========================================================
    # LOOP OVER TIME WITH PROGRESS BAR
    # ========================================================

    for t in tqdm(
        range(a_ice.sizes["time"]),
        desc="Computing FESOM ice edge",
        unit="timestep"
    ):

        # Read SIC for current time step
        sic = np.asarray(
            a_ice.isel(time=t).values
        )

        # Mask invalid nodes and nodes south of min_lat
        bad_nodes = (
            (~np.isfinite(sic))
            | (~north_mask)
        )

        # Mask triangles touching invalid/southern nodes
        bad_triangles = np.any(
            bad_nodes[triangles],
            axis=1
        )

        tri.set_mask(bad_triangles)

        # ====================================================
        # EXTRACT 15% SIC CONTOUR
        # ====================================================

        fig, ax = plt.subplots()

        cs = ax.tricontour(
            tri,
            sic,
            levels=[level]
        )

        plt.close(fig)

        lons_all = []
        lats_all = []

        # ====================================================
        # PROCESS CONTOUR SEGMENTS
        # ====================================================

        for seg in cs.allsegs[0]:

            lon_seg = seg[:, 0]
            lat_seg = seg[:, 1]

            lon_seg = (
                (lon_seg + 180) % 360
            ) - 180

            good = (
                np.isfinite(lon_seg)
                & np.isfinite(lat_seg)
            )

            lon_seg = lon_seg[good]
            lat_seg = lat_seg[good]

            if len(lon_seg) > 2:

                order = np.argsort(lon_seg)

                lon_seg = lon_seg[order]
                lat_seg = lat_seg[order]

                lons_all.append(lon_seg)
                lats_all.append(lat_seg)

        # ====================================================
        # INTERPOLATE ICE EDGE TO LONGITUDE GRID
        # ====================================================

        edge_lat = np.full(
            len(lon_grid),
            np.nan
        )

        for i, lon0 in enumerate(lon_grid):

            candidates = []

            for lon_seg, lat_seg in zip(
                lons_all,
                lats_all
            ):

                if (
                    lon0 >= np.nanmin(lon_seg)
                    and lon0 <= np.nanmax(lon_seg)
                ):

                    lat0 = np.interp(
                        lon0,
                        lon_seg,
                        lat_seg
                    )

                    candidates.append(lat0)

            if len(candidates) > 0:

                # Southernmost crossing
                edge_lat[i] = np.nanmin(
                    candidates
                )

        edge_all_times.append(edge_lat)

    # ========================================================
    # CREATE XARRAY DATAARRAY
    # ========================================================

    edge_da = xr.DataArray(
        np.array(edge_all_times),
        dims=("time", "longitude"),
        coords={
            "time": a_ice["time"],
            "longitude": lon_grid
        },
        name="fesom_ice_edge_latitude",
        attrs={
            "description": (
                "Latitude where FESOM sea ice "
                "concentration crosses threshold"
            ),
            "units": "degrees_north",
            "threshold": level,
            "threshold_units": "fraction",
            "minimum_latitude_used": min_lat
        }
    )

    return edge_da


# ============================================================
# 2. RUN CALCULATION
# ============================================================

start = time.time()

fesom_edge = ice_edge_latitude_by_lon_fesom(
    a_ice,
    mesh,
    level=0.15,
    min_lat=60
)

end = time.time()

# ============================================================
# 3. RESULTS
# ============================================================

print("\nFESOM ice-edge calculation completed!")
print(f"Execution time: {end - start:.3f} seconds")
print(f"Number of time steps: {fesom_edge.sizes['time']}")
print(f"Number of longitudes: {fesom_edge.sizes['longitude']}")

print("\nResult:")
print(fesom_edge)

In [ ]:
#saving the fesom data
np.save("./saved_data/fesom_ice_edge_2014.npy", fesom_edge.values)



In [ ]:
#loading fesom data back
fesom_edge = np.load("./saved_data/fesom_ice_edge_2014.npy")

print("Loaded FESOM ice edge:", fesom_edge.shape)

In [ ]:
#plot the loaded data (copernicus and fesom)

lon = np.arange(-180, 181)
day = 100  # January 1

plt.figure(figsize=(10, 4))

plt.plot(lon, ice_edge[day], label="Copernicus")
plt.plot(lon, fesom_edge[day], label="FESOM2")

plt.xlabel("Longitude (°)")
plt.ylabel("Ice-edge latitude (°N)")
plt.title("15% sea-ice edge — 1 January 2014")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# FESOM2 - Copernicus
assert fesom_edge.shape == ice_edge.shape, "Array shapes differ!"

diff_values = fesom_edge - ice_edge

diff_edge = xr.DataArray(
    diff_values,
    dims=("time", "longitude"),
    coords={
        "time": np.arange(diff_values.shape[0]),
        "longitude": np.arange(-180, 181, 1)
    },
    name="fesom_minus_osisaf_edge"
)

print(diff_edge)

In [ ]:
# plotting the raw data
cmap = plt.cm.RdBu_r.copy()
cmap.set_bad(color="lightgray")

finite = np.isfinite(diff_edge.values)

vmax = np.nanmax(np.abs(diff_edge.values[finite]))
vmin = -vmax

plt.figure(figsize=(14, 6))

plt.pcolormesh(
    diff_edge.longitude,
    diff_edge.time,
    diff_edge,
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    shading="auto"
)

cbar = plt.colorbar()
cbar.set_label("Ice-edge latitude difference [degrees]\nFESOM - OSI-SAF")

plt.xlabel("Longitude")
plt.ylabel("January day index")
plt.title("Difference in 15% sea-ice edge latitude: FESOM - OSI-SAF")

plt.tight_layout()
plt.show()

In [ ]:
# calcualting monthly means

# Assign daily dates for 2014
diff_edge = diff_edge.assign_coords(
    time=pd.date_range("2014-01-01", periods=365, freq="D")
)

# Calculate monthly means
diff_monthly = diff_edge.resample(time="MS").mean()

print("Daily shape:", diff_edge.shape)
print("Monthly shape:", diff_monthly.shape)

In [ ]:
# Number of days in each month
days_in_month = diff_monthly.time.dt.days_in_month

# Require at least 50% valid days
min_valid = np.ceil(0.5 * days_in_month)

# Mask insufficiently sampled monthly means
diff_monthly_filtered = diff_monthly.where(
    valid_days >= min_valid
)

In [ ]:
# plot monthly averaged data

cmap = plt.cm.RdBu_r.copy()
cmap.set_bad(color="lightgray")

# Symmetric colour limits
finite = np.isfinite(diff_monthly.values)

vmax = np.nanmax(np.abs(diff_monthly.values[finite]))
vmin = -vmax

month_names = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

y = np.arange(12)

fig, ax = plt.subplots(figsize=(14, 6))

pcm = ax.pcolormesh(
    diff_monthly.longitude,
    y,
    diff_monthly_filtered.values,
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    shading="auto"
)

cbar = plt.colorbar(pcm, ax=ax)
cbar.set_label(
    "Ice-edge latitude difference [degrees]\nFESOM - OSI-SAF"
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Month")
ax.set_title(
    "Monthly mean difference in 15% sea-ice edge latitude: FESOM - OSI-SAF"
)

ax.set_yticks(y)
ax.set_yticklabels(month_names)

plt.tight_layout()

plt.savefig(
    "SIC_comparison_monthly.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:

# ============================================================
# 1. 10-DEGREE LONGITUDE BINNING
# ============================================================

lon_bins = np.arange(-180, 181, 10)
lon_centers = lon_bins[:-1] + 5

# Use FILTERED monthly data
diff_binned = diff_monthly_filtered.groupby_bins(
    "longitude",
    bins=lon_bins,
    labels=lon_centers,
    include_lowest=True
).mean()

# Rename coordinate
diff_binned = diff_binned.rename({
    "longitude_bins": "longitude"
})

print("Binned data shape:", diff_binned.shape)

# ============================================================
# 2. COLOUR SCALE
# ============================================================

cmap = plt.cm.RdBu_r.copy()
cmap.set_bad(color="lightgray")

finite = np.isfinite(diff_binned.values)

vmax = np.nanmax(np.abs(diff_binned.values[finite]))
vmin = -vmax

# ============================================================
# 3. PLOT
# ============================================================

month_names = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

y = np.arange(diff_binned.sizes["time"])

fig, ax = plt.subplots(figsize=(14, 6))

pcm = ax.pcolormesh(
    diff_binned.longitude,
    y,
    diff_binned.values,
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    shading="auto"
)

cbar = plt.colorbar(pcm, ax=ax)
cbar.set_label(
    "Ice-edge latitude difference [degrees]\nFESOM - OSI-SAF"
)

ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Month")
ax.set_title(
    "Monthly mean 15% sea-ice edge difference: FESOM - OSI-SAF (10° bins)"
)

ax.set_yticks(y)
ax.set_yticklabels(month_names)

plt.tight_layout()

plt.savefig(
    "SIC_comparison_monthly_10deg_bins.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# adding the secotr map

# ============================================================
# Use monthly filtered data with 10-degree longitude bins
# ============================================================

# diff_binned was calculated from diff_monthly_filtered
print("Binned data shape:", diff_binned.shape)
assert diff_binned.shape == (12, 36)

# consistent sectors with Schulz
# -----------------------------
# Font sizes
# -----------------------------
TITLE_FONTSIZE = 22
LABEL_FONTSIZE = 20
TICK_FONTSIZE = 18
CBAR_FONTSIZE = 18
INSET_TITLE_FONTSIZE = 14

# -----------------------------
# Sector-boundary guide lines
# -----------------------------
guide_lons = [-140, -80, -40, 20, 80, 170]

guide_colors = [
    "#CC79A7",  # CS
    "#E69F00",  # BB
    "#009E73",  # AS
    "#0072B2",  # BS
    "#D55E00",  # ESS
    "#D55E00",  # ESS
]

# -----------------------------
# Colormap
# -----------------------------
cmap = plt.cm.RdBu_r.copy()
cmap.set_bad(color="lightgray")

finite = np.isfinite(diff_binned.values)
vmax = np.nanmax(np.abs(diff_binned.values[finite]))
vmin = -vmax

# -----------------------------
# Month axis: 1–12 instead of 0–11
# -----------------------------
y = np.arange(1, diff_binned.sizes["time"] + 1)

# -----------------------------
# Main plot
# -----------------------------
fig, ax = plt.subplots(figsize=(14, 6))

pcm = ax.pcolormesh(
    diff_binned.longitude,
    y,
    diff_binned.values,
    cmap=cmap,
    vmin=-15,
    vmax=15,
    shading="auto"
)

# -----------------------------
# Sector-boundary guide lines
# -----------------------------
for lon0, col in zip(guide_lons, guide_colors):
    ax.axvline(
        lon0,
        color=col,
        linewidth=2,
        linestyle="--",
        alpha=0.9
    )

# -----------------------------
# Colorbar
# -----------------------------
cbar = plt.colorbar(pcm, ax=ax)

cbar.set_label(
    "Ice-edge latitude difference (°)\nmodel − obs",
    fontsize=CBAR_FONTSIZE
)

cbar.ax.tick_params(labelsize=TICK_FONTSIZE)

# -----------------------------
# Labels and title
# -----------------------------
ax.set_xlabel("Longitude", fontsize=LABEL_FONTSIZE)
ax.set_ylabel("Month", fontsize=LABEL_FONTSIZE)

ax.set_title(
    "Difference in 15% sea-ice edge latitude: Model - Obs",
    fontsize=TITLE_FONTSIZE,
)

ax.set_yticks(np.arange(1, 13))

ax.tick_params(
    axis="both",
    labelsize=TICK_FONTSIZE
)

# -----------------------------
# Inset Arctic map
# -----------------------------
axins = fig.add_axes(
    [0.545, 0.15, 0.25, 0.25],
    projection=ccrs.NorthPolarStereo()
)

axins.set_extent(
    [-180, 180, 50, 90],
    crs=ccrs.PlateCarree()
)

axins.add_feature(
    cfeature.OCEAN,
    facecolor="white",
    zorder=0
)

axins.add_feature(
    cfeature.LAND,
    facecolor="lightgray",
    edgecolor="black",
    linewidth=0.4,
    zorder=1
)

axins.coastlines(linewidth=0.4)

# Draw matching sector-boundary lines on inset map
lat_line = np.linspace(50, 90, 200)

for lon0, col in zip(guide_lons, guide_colors):
    axins.plot(
        np.full_like(lat_line, lon0),
        lat_line,
        color=col,
        linewidth=1.8,
        transform=ccrs.PlateCarree(),
        zorder=3
    )

axins.set_title(
    "Sector boundaries",
    fontsize=INSET_TITLE_FONTSIZE,
    fontweight="bold",
    pad=2
)

axins.spines["geo"].set_linewidth(0.5)

plt.tight_layout()

plt.savefig(
    "SIC_comparison_map_JRA55.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()